# 

# Load Dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/raw/bank-marketing.csv')
df.head()

,age,age group,eligible,job,salary,marital,education,marital-education,targeted,default,...,contact,day,month,duration,campaign,pdays,previous,poutcome,y,response
0,58,5,Y,management,100000,married,tertiary,married-tertiary,yes,no,...,unknown,5,may,261,1,-1,0,unknown,no,0
1,44,4,Y,technician,60000,single,secondary,single-secondary,yes,no,...,unknown,5,may,151,1,-1,0,unknown,no,0
2,33,3,Y,entrepreneur,120000,married,secondary,married-secondary,yes,no,...,unknown,5,may,76,1,-1,0,unknown,no,0
3,47,4,Y,blue-collar,20000,married,unknown,married-unknown,no,no,...,unknown,5,may,92,1,-1,0,unknown,no,0
4,33,3,Y,unknown,0,single,unknown,single-unknown,no,no,...,unknown,5,may,198,1,-1,0,unknown,no,0


## Data Cleaning

In [3]:
df.replace('unknown', np.nan, inplace=True)

for col in ['job', 'education', 'contact', 'poutcome']:
    df[col] = df[col].fillna(df[col].mode()[0])

In [4]:
df.isnull().sum()

age                  0
age group            0
eligible             0
job                  0
salary               0
marital              0
education            0
marital-education    0
targeted             0
default              0
balance              0
housing              0
loan                 0
contact              0
day                  0
month                0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
y                    0
response             0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   age                45211 non-null  int64 
 1   age group          45211 non-null  int64 
 2   eligible           45211 non-null  object
 3   job                45211 non-null  object
 4   salary             45211 non-null  int64 
 5   marital            45211 non-null  object
 6   education          45211 non-null  object
 7   marital-education  45211 non-null  object
 8   targeted           45211 non-null  object
 9   default            45211 non-null  object
 10  balance            45211 non-null  int64 
 11  housing            45211 non-null  object
 12  loan               45211 non-null  object
 13  contact            45211 non-null  object
 14  day                45211 non-null  int64 
 15  month              45211 non-null  object
 16  duration           45211 non-null  int64

# Integrar FEAST

In [6]:
## Inicializar repositorio Feast
# !cd .. && uv run feast init bank_marketing_feast

## Registrar feature
# !cd ../bank_marketing_feast/feature_repo && uv run feast apply

## Explorar data en la web UI
# !cd ../bank_marketing_feast/feature_repo && uv run feast ui --port 7001

## Instalación

In [7]:
!uv add feast

Resolved 79 packages in 5ms
Audited 78 packages in 0.07ms


In [8]:
# Verficar instalación de feast
!uv run python -c "import feast; print(feast.__version__)"

0.56.0


## Guardar dataset para la ingesta

In [9]:
from pathlib import Path

PROJ_ROOT = Path.cwd().parent
DATA_ROOT = f"{PROJ_ROOT}/bank_marketing_feast/feature_repo/data"

df["customer_id"] = range(1, len(df) + 1)
df["event_timestamp"] = pd.Timestamp.now()
df.to_parquet(f"{DATA_ROOT}/bank_marketing.parquet")

## Registrar features

In [10]:
from feast import FeatureStore

store = FeatureStore(repo_path="../bank_marketing_feast/feature_repo")

# Aplicar el repositorio (registrar las features)
store.apply(["../bank_marketing_feast/feature_repo/feature_view.py"])


## Consultar las features desde Feast

In [11]:
# Obtener features para entrenamiento
feature_vector = store.get_historical_features(
    entity_df=df[["customer_id", "event_timestamp"]],
    features=[
        "bank_features:age",
        "bank_features:balance",
        "bank_features:duration",
        "bank_features:campaign",
        "bank_features:previous",
        "bank_features:job",
        "bank_features:marital",
        "bank_features:education",
        "bank_features:default",
        "bank_features:housing",
        "bank_features:loan",
        "bank_features:contact",
        "bank_features:month",
        "bank_features:poutcome",
    ],
).to_df()

feature_vector.head()


,customer_id,event_timestamp,age,balance,duration,campaign,previous,job,marital,education,default,housing,loan,contact,month,poutcome
0,1,2025-11-10 19:09:12.766960+00:00,58,2143,261,1,0,management,married,tertiary,no,yes,no,cellular,may,failure
1,30136,2025-11-10 19:09:12.766960+00:00,52,941,89,2,0,admin.,married,secondary,no,no,no,cellular,feb,failure
2,30137,2025-11-10 19:09:12.766960+00:00,33,315,146,3,0,blue-collar,married,secondary,no,no,no,cellular,feb,failure
3,30138,2025-11-10 19:09:12.766960+00:00,53,1355,447,2,8,management,divorced,secondary,no,no,yes,cellular,feb,other
4,30139,2025-11-10 19:09:12.766960+00:00,28,392,214,2,2,admin.,single,secondary,no,yes,yes,cellular,feb,failure


# Genearar dataset para la etapa de preprocesamiento

In [14]:
feature_vector = feature_vector.merge(df[["customer_id", "y"]], on="customer_id", how="left")

In [17]:
feature_vector.head()

,customer_id,event_timestamp,age,balance,duration,campaign,previous,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,1,2025-11-10 19:09:12.766960+00:00,58,2143,261,1,0,management,married,tertiary,no,yes,no,cellular,may,failure,no
1,30136,2025-11-10 19:09:12.766960+00:00,52,941,89,2,0,admin.,married,secondary,no,no,no,cellular,feb,failure,no
2,30137,2025-11-10 19:09:12.766960+00:00,33,315,146,3,0,blue-collar,married,secondary,no,no,no,cellular,feb,failure,yes
3,30138,2025-11-10 19:09:12.766960+00:00,53,1355,447,2,8,management,divorced,secondary,no,no,yes,cellular,feb,other,no
4,30139,2025-11-10 19:09:12.766960+00:00,28,392,214,2,2,admin.,single,secondary,no,yes,yes,cellular,feb,failure,no


In [27]:
feature_vector_sorted = feature_vector.sort_values(by="customer_id")

In [30]:
feature_vector_sorted.to_csv(f"{PROJ_ROOT}/data/processed/feast-bank-marketing.csv", index=False)